# 03 · Red personajes ↔ lugares

Detecta personajes en cada párrafo del corpus y construye la red bipartita de coocurrencia entre personajes y lugares.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import corpus, spatial_extraction, character_extraction
import exporters, viz_network, interactive

cfg = corpus.load_config('configs/sin_remedio.yaml')
parrafos = corpus.parse_from_config('data/corpus.txt', cfg)

## 1. Cargar catálogos

In [ ]:
with open(cfg['lugares_catalogo_path'], encoding='utf-8') as f:
    catalogo_lugares = json.load(f)
with open(cfg['personajes_catalogo_path'], encoding='utf-8') as f:
    catalogo_personajes = json.load(f)

print(f"Lugares: {len(catalogo_lugares)} · Personajes: {len(catalogo_personajes)}")

## 2. Detectar personajes en cada párrafo

In [ ]:
pers_idx = character_extraction.index_paragraphs(parrafos, catalogo_personajes)

n_con = sum(1 for s in pers_idx.values() if s)
print(f"Párrafos con al menos un personaje: {n_con} / {len(pers_idx)}")

## 3. Cruzar con la extracción espacial

In [ ]:
df_esp = spatial_extraction.extract(parrafos, catalogo_lugares)
df_co = character_extraction.cooccurrence_with_spatial(df_esp, pers_idx, catalogo_personajes)
co_counts = character_extraction.cooccurrence_counts(df_co)
co_counts.head(15)

## 4. Visualización: grafo bipartito

In [ ]:
pers_info = {p['canonico']: p for p in catalogo_personajes}

viz_network.bipartite_network(
    co_counts,
    pers_info,
    cfg['categorias_sociales'],
    'outputs/red_bipartita.png',
    min_personaje_weight=3,
)

## 5. Exportar datos del grafo

In [ ]:
# Edges en formato source/target/weight para Gephi, D3…
edges = co_counts.rename(columns={'Personaje':'source','Lugar':'target','Coocurrencias':'weight'})
exporters.to_csv(edges, 'outputs/red_edges.csv')

# JSON estructurado para D3
nodes = []
for p in catalogo_personajes:
    n_apar = (co_counts['Personaje']==p['canonico']).sum()
    if n_apar==0: continue
    nodes.append({'id':f"P:{p['canonico']}",'name':p['canonico'],'type':'personaje',
                  'circulo':p['circulo'],'weight':int(co_counts[co_counts['Personaje']==p['canonico']]['Coocurrencias'].sum())})

for lugar in co_counts['Lugar'].unique():
    info = next((c for c in catalogo_lugares if c['nombre_canonico']==lugar), None)
    nodes.append({'id':f"L:{lugar}",'name':lugar,'type':'lugar',
                  'lat':info.get('lat') if info else None,
                  'lon':info.get('lon') if info else None,
                  'weight':int(co_counts[co_counts['Lugar']==lugar]['Coocurrencias'].sum())})

edges_json = [{'source':f"P:{e['Personaje']}",'target':f"L:{e['Lugar']}",'weight':int(e['Coocurrencias'])}
              for _,e in co_counts.iterrows()]

graph = {'nodes':nodes,'edges':edges_json}
exporters.to_json_records([graph], 'outputs/grafo.json')
print(f'Nodos: {len(nodes)} · Aristas: {len(edges_json)}')

## 6. HTML interactivo

In [ ]:
# Construir GeoJSON con personajes por lugar
features = []
for lugar in co_counts['Lugar'].unique():
    info = next((c for c in catalogo_lugares if c['nombre_canonico']==lugar), None)
    if not info or info.get('lat') is None: continue
    pers_aqui = co_counts[co_counts['Lugar']==lugar][['Personaje','Coocurrencias']]
    features.append({
        'type':'Feature',
        'geometry':{'type':'Point','coordinates':[info['lon'],info['lat']]},
        'properties':{
            'name':lugar,
            'total_coocurrencias':int(pers_aqui['Coocurrencias'].sum()),
            'personajes':[{'nombre':r['Personaje'],'apariciones':int(r['Coocurrencias'])} for _,r in pers_aqui.iterrows()],
        }
    })

interactive.network_map_html(graph, {'type':'FeatureCollection','features':features},
                              'outputs/mapa_red_interactivo.html')
print('✓ HTML interactivo de red generado')